# Task 1

Hentet kode fra frosty nettside linje code box [1]-[3]


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import requests

with open("client_id.txt", "r") as file:
    client_id = file.read().strip().split(":")[1].strip()


In [ ]:
endpoint = 'https://frost.met.no/observations/v0.jsonld'
parameters = {
    'sources': 'SN17850',
    'elements': 'mean(air_temperature P1D)',
    'referencetime': '2025-01-01/2025-12-31',
}

# Issue an HTTP GET request
r = requests.get(endpoint, parameters, auth=(client_id,''))
# Extract JSON data
json = r.json()

In [ ]:
# Check if the request worked, print out any errors
if r.status_code == 200:
    data = json['data']
    print('Data retrieved from frost.met.no!')
else:
    print('Error! Returned status code %s' % r.status_code)
    print('Message: %s' % json['error']['message'])
    print('Reason: %s' % json['error']['reason'])


Tok inspirasjon fra Frosty code men den var utdatert box[4]

In [ ]:
rows = []
for item in data:
    observation = item['observations'][0]
    rows.append({
        'date': pd.to_datetime(item['referenceTime']).tz_localize(None).normalize(),
        'source_Id': item['sourceId'],
        'element_Id': observation['elementId'],
        'temperature': observation['value'],
        'unit': observation['unit'],
        'time_offset': observation['timeOffset']
    })
df = pd.DataFrame(rows).set_index('date')

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5)) 
ax.plot(df.index,df['temperature'])
ax.set_title('Daglig middeltemperatur, Ås 2025')
ax.set_xlabel('Dato')
ax.set_ylabel('Temperatur (grader celsius)')
ax.grid()
plt.show()


In [ ]:
print('Sammendrag av lufttemperatur, Ås 2025')
print('-' * len('Sammendrag av lufttemperatur, Ås 2025'))
print(f"{'Gjennomsnitt':<15}{df['temperature'].mean():>8.1f} °C")
print(f"{'Median':<15}{df['temperature'].median():>8.1f} °C")
print(f"{'Min':<15}{df['temperature'].min():>8.1f} °C")
print(f"{'Maks':<15}{df['temperature'].max():>8.1f} °C")

# Task 2

Laget en def for å kunne gjenbruke i oppgave 3 og 4 slik at det ikke blir så mye repitisjon

In [ ]:
def get_frost_periods(source_id, element, start, end, client_id):
    r = requests.get('https://frost.met.no/observations/v0.jsonld',
                     {'sources': source_id,
                      'elements': element,
                      'referencetime': f'{start}/{end}'},
                      auth=(client_id,''))
    if r.status_code != 200:
        err = r.json()['error']
        raise RuntimeError(f"{r.status_code}: {err['message']} - {err['reason']}")
    return pd.Series(
        {pd.to_datetime(d['referenceTime']).tz_localize(None).normalize():
         d['observations'][0]['value'] for d in r.json()['data']}
    ).sort_index()

precipitation = get_frost_periods('SN17850', 'sum(precipitation_amount P1D)', '2025-01-01', '2025-12-31', client_id)

In [ ]:
temperature = df['temperature']
fig, ax1 = plt.subplots(figsize=(12, 5)) 
ax1.bar(precipitation.index, precipitation.values)
ax1.set_xlabel('Dato')
ax1.set_ylabel('Nedbør (mm/døgn)')

ax2 = ax1.twinx()
ax2.plot(temperature.index, temperature.values, color='tab:red', label='Middeltemperatur (°C)')
ax2.set_ylabel('Temperatur (°C)', color='tab:red')

ax1.set_title('Temperatur og nedbør, Ås 2025')
ax1.grid()
plt.show()


# Task 3

In [ ]:
old = get_frost_periods('SN17850', 'sum(air_temperature PID)', '1925-01-01', '1925-12-31', client_id)
new = df['temperature']

fig, ax = plt.subplots(figsize=(12,5))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

def to_common_year(s):
    s.copy()
    s = pd.to_datetime({'year':2001})